In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import OneHotEncoder
import math

!pip install mlxtend==0.23.1

# Association Rule for Store Dataset

In this case study, we will explore how association rule can be used to analyze the items that are usualy purcased together.

you can refer to this article to find out about apriori and association rule:
https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/
https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/

## Load Data

We will use the dataset of the transaction in a certain store. You can get the dataset here:
https://gist.githubusercontent.com/Harsh-Git-Hub/2979ec48043928ad9033d8469928e751/raw/72de943e040b8bd0d087624b154d41b2ba9d9b60/retail_dataset.csv

In [ ]:
# load the data set ans show the first five transaction
df = pd.read_csv('https://gist.githubusercontent.com/Harsh-Git-Hub/2979ec48043928ad9033d8469928e751/raw/72de943e040b8bd0d087624b154d41b2ba9d9b60/retail_dataset.csv')
df.head()

# Get the set of product that has been purchased


In [ ]:
# Get the set of purchased products (including NaN)
purchased_products = set(item for row in df.values for item in row)

# Sort the set alphabetically, keeping NaN in its position
sorted_products = sorted(purchased_products, key=lambda x: (str(x) if pd.notna(x) else 'zzzzz'))

# Output the result
print(sorted_products)

## Preprocess Data

In this step, we will transform our dataset so that we will have a one hot encoding based on the purchased products.

In [ ]:
# Create an itemset that includes NaN values
itemset = set(item for row in df.values for item in row)

# Encoding the features (one-hot encoding)
encoded_rows = []  # List to hold the encoded rows

for row in df.values:
    # Create a dictionary with items as keys, and 1 or 0 as values depending on presence in the row
    row_encoded = {item: 1 if item in row else 0 for item in itemset}
    encoded_rows.append(row_encoded)

# Example: Output the encoded row as a dictionary for the first row
print(encoded_rows[0])

In [ ]:
# Create a new DataFrame from the encoded rows
new_df = pd.DataFrame(encoded_rows)

# Show the new dataframe
new_df.head()

In [ ]:
## Drop the NaN column if it exists (assuming it's the last column or contains only NaN values)
new_df = new_df.loc[:, new_df.columns.notna()]

# Show the new dataframe
print(new_df.head())

Since, the encoded dataframe consist of the empty column. We will drop the NaN column or select all columns other than the first column.

## Apriori Algorithm

We will use appriori algorithm to determine the frequently purchased products.
For this case study, we will min_support=0.2

In [ ]:
#Set threshold value untuk digunakan dalam penghitungan support
from mlxtend.frequent_patterns import apriori, association_rules
# Apply Apriori Algorithm with min_support=0.2
frequent_itemsets = apriori(new_df, min_support=0.2, use_colnames=True)

# Display frequent itemsets
print(frequent_itemsets)

The we will generate association rule of the frequent itemset based on confidence level with the threshold=0.6

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules.drop(columns=['zhangs_metric'], inplace=True, errors='ignore')
rules

Provide explanation about __antecedent support__, __consequent support__, __support__, __confidence__, __lift__, __leverage__, __conviction__, __conviction__ and the interpretation from the case above (please use text section)

- Antecedent Support: The proportion of transactions that contain the antecedent (left-hand item). For example, "Wine" has an antecedent support of 43.81%, meaning it appears in 43.81% of all transactions.

- Consequent Support: The proportion of transactions that contain the consequent (right-hand item). For "Cheese," the consequent support is 50.16%, indicating that it appears in 50.16% of the transactions.

- Support: The proportion of transactions where both the antecedent and consequent appear together. For the rule {Wine} => {Cheese}, the support is 26.98%, meaning both items are found together in 26.98% of transactions.

- Confidence: Confidence is a measure of the reliability or strength of an association rule. It quantifies the likelihood that if itemset X is purchased, itemset Y will also be purchased. The likelihood that the consequent is purchased when the antecedent is purchased. For the rule {Wine} => {Cheese}, the confidence is 61.59%, meaning that 61.59% of transactions that contain "Wine" also contain "Cheese."

- Lift: Lift is a metric that measures the strength of association between two itemsets (A and B) while taking into account the expected or random occurrence of B given A. Measures how much more likely the consequent is to appear when the antecedent is purchased, compared to when they appear independently. A lift of 1.23 for the rule {Wine} => {Cheese} suggests that the occurrence of "Cheese" is 1.23 times more likely when "Wine" is bought, compared to when it is bought independently.

- Leverage: Leverage measures the difference between the observed frequency of co-occurrence of two items (X and Y) in transactions and the frequency that would be expected if X and Y were independent of each other. Shows the difference between the observed support and the expected support if the antecedent and consequent were independent. For the rule {Wine} => {Cheese}, the leverage is 0.05, indicating a slight positive relationship between the two items appearing together.

- Conviction: Conviction measures the ratio of the expected frequency that X occurs without Y to the observed frequency that X occurs without Y. Measures the reliability of the rule. Higher conviction values (close to 1) indicate that the rule is more reliable in predicting the consequent. For {Wine} => {Cheese}, the conviction is 1.30, meaning the rule is relatively reliable.

In summary, the rules with higher confidence, lift, leverage, and conviction suggest strong and reliable associations, while those with lower values suggest weaker relationships between the antecedents and consequents.


## References:
https://chatgpt.com/share/6748369a-f208-8010-8285-2184f0a7864f

https://medium.com/@iuyasik/market-basket-analysis-apriori-algorithm-using-zhangs-metric-708406fc5dfc
